# FDIC banks: pre-explore, reason, then measure

Deposit funding cost is a bank-level quantity with a clear economic reading:
how many percentage points of interest a bank pays, on average, on its
interest-bearing deposits. Aggregate series in
[](deposit_elasticity.ipynb) and [](deposit_saturation.ipynb) already show
how that cost moves with the offered rate. This case stays at the
**institution-year** panel and asks a different question: after the calendar
rate environment is controlled, which recorded bank traits help predict the
rate actually paid — and how an analyst should move from automated probes to
a measured `SearchSession`.

The analysis uses four steps:

1. **Exhaustive native probes** (`pre_scan` with `native` / `neural` / `dual`)
2. **Dissection** of variables, shapes, and nominated interactions
3. **Domain reasoning** (symptoms vs causes; monotonic effects with local bumps)
4. **Measured search** (`compare` / `adopt` / `tune` / `freeze` / `final_evaluate`)

The mboostrs probes propose variables and terms. Candidate specifications
are then compared on shared validation folds before any adoption.


:::{admonition} What this study estimates
:class: dropdown

| | |
|---|---|
| **Target** | which bank traits predict deposit funding cost once the Fed cycle is controlled, using probes only as screening |
| **Observed response** | `deposit_rate` = \(100 \times \mathrm{EDEP}/\mathrm{DEPI}\), percent paid on interest-bearing deposits |
| **Estimand** | conditional mean deposit rate given bank structure and calendar year (predictive association, not a causal funding-cost elasticity) |
| **Units** | percentage points per year |
| **Sampling unit** | one insured US bank in one December Call Report year (`CERT` × `REPDTE`) |
| **Dependence** | banks repeat across years; the same franchise appears in several rows |
| **Split unit** | ~18% of banks held out entirely for one-shot `final_evaluate`; training folds are ordinary 4-fold CV on the remaining bank-years |
| **Tuning** | `tune_mstop` on the accepted specification after measured adopts |
| **Uncertainty** | paired fold standard errors on specification `change`; adopt only when \(\lvert\Delta\rvert > 2\,\mathrm{SE}\) |
| **Claim type** | predictive / methodological — demonstrates the preexplore→reason→measure loop on real FDIC data |

:::


## Tree, shape and neural screening calls

| Role | mboostrs call | What it does here |
|------|---------------|-------------------|
| Boosted-tree probe | `session.run("pre_scan", engine="native")` | permutation importance, partial-dependence shape hints, Friedman \(H^2\) pairs |
| Neural probe | `session.run("pre_scan", engine="neural")` | native neural nomination of importance and pairs |
| Agreement | `session.run("pre_scan", engine="dual")` | tree + neural; `pair_sources` marks agreement |
| Shape / MARS-like screen | `pre_scan` partial `linear_r2` / `curve_r2` / `step_share`, then `term_report(..., shapes=True)` | line vs spline vs stump alternatives measured out of fold |
| Residual interactions | `interaction_screen`, `pair_candidates` | residual and surrogate pair terms (`bbs %X% bbs`, …) |
| Multiway tree nomination | `tree_candidate` | one `btree(...)` over accepted columns |
| Decision | `compare` → `adopt` → `tune` → `freeze` → `final_evaluate` | only measured risk can change the specification |

A live transcript of this loop is checked in as
`bench_fdic_preexplore_reason.json` (built by
`tools/run_fdic_preexplore_reason.py`).


In [1]:
from pathlib import Path
import json
import numpy as np
import polars as pl
from IPython.display import display

import mboostrs as mb
from mboostrs.policy import CostBudget
from mboostrs.search_session import open_search

TRANSCRIPT = next((path for path in (
    Path("bench_fdic_preexplore_reason.json"),
    Path("cases/bench_fdic_preexplore_reason.json"),
    Path("docs/book/cases/bench_fdic_preexplore_reason.json"),
) if path.is_file()), None)
if TRANSCRIPT is None:
    raise FileNotFoundError("The frozen FDIC transcript is missing")


In [2]:
# Prepare one bank-year response and the candidate funding predictors.
raw = mb.datasets.fdic_bank_yearend(output="polars")
panel = (
    raw.drop_nulls(["DEP", "EDEP", "ASSET", "DEPI", "LNLSNET", "BRO", "OFFDOM"])
    .filter((pl.col("DEP") > 0) & (pl.col("ASSET") > 0) & (pl.col("DEPI") > 0) & (pl.col("EDEP") >= 0))
    .with_columns([
        pl.col("REPDTE").str.slice(0, 4).cast(pl.Int32).alias("year"),
        (100.0 * pl.col("EDEP") / pl.col("DEPI")).alias("deposit_rate"),
        pl.col("ASSET").log1p().alias("log_assets"),
        (pl.col("LNLSNET") / pl.col("ASSET")).alias("loan_ratio"),
        (pl.col("BRO") / pl.col("DEP")).clip(0, 1).alias("brokered_share"),
        (pl.col("DEP") / pl.col("ASSET")).alias("deposit_funding"),
        (pl.col("DEPI") / pl.col("DEP")).alias("ib_share"),
        pl.col("OFFDOM").clip(0, 200).alias("offices"),
    ])
    .filter((pl.col("deposit_rate") > 0.05) & (pl.col("deposit_rate") < 8.0))
)
print(f"panel banks×years: {panel.height:,} rows, {panel['CERT'].n_unique():,} banks, "
      f"years {panel['year'].min()}–{panel['year'].max()}")
display(panel.select(["deposit_rate", "log_assets", "brokered_share", "offices", "loan_ratio"]).describe())

panel banks×years: 43,906 rows, 5,727 banks, years 2017–2025


statistic,deposit_rate,log_assets,brokered_share,offices,loan_ratio
str,f64,f64,f64,f64,f64
"""count""",43906.0,43906.0,43906.0,43906.0,43906.0
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",1.162378,12.744764,0.032438,10.33287,0.629239
"""std""",0.929874,1.503591,0.080112,24.530362,0.167504
"""min""",0.050034,7.971776,0.0,0.0,0.0
"""25%""",0.478775,11.75823,0.0,2.0,0.53321
"""50%""",0.828758,12.570091,0.0,4.0,0.659149
"""75%""",1.634336,13.482595,0.03175,9.0,0.754928
"""max""",7.875752,22.045731,0.999895,200.0,0.991474


## FDIC bank-year panel

`mb.datasets.fdic_bank_yearend()` loads the curated FDIC BankFind
year-end extract already used behind the deposit cases: assets, deposits,
interest expense on deposits (`EDEP`), brokered deposits, loans, and domestic
offices for every insured bank each December from 2017 through 2025. The panel contains repeated measurements of bank size, lending and funding
structure, allowing us to compare their associations across rate environments.

The response `deposit_rate = 100 × EDEP / DEPI` is the average rate paid on
interest-bearing deposits. It is the bank-level analogue of the funding cost
in [](deposit_saturation.ipynb), observed across thousands of franchises
rather than one aggregate path.


In [3]:
# Stratified subsample + bank holdout (same design as the checked-in transcript).
PREDICTORS = ["year", "log_assets", "loan_ratio", "brokered_share",
              "deposit_funding", "ib_share", "offices"]
COLS = ["deposit_rate", "CERT", *PREDICTORS]

pieces = []
for year in panel["year"].unique().sort().to_list():
    block = panel.filter(pl.col("year") == year).select(COLS)
    pieces.append(block.sample(n=min(700, block.height), seed=year))
sample = pl.concat(pieces)

banks = np.array(sample["CERT"].unique().sort().to_list(), dtype=np.int64)
rng = np.random.default_rng(7)
rng.shuffle(banks)
n_test = max(1, int(round(0.18 * len(banks))))
test_banks = set(banks[:n_test].tolist())
train = sample.filter(~pl.col("CERT").is_in(list(test_banks))).drop("CERT")
test = sample.filter(pl.col("CERT").is_in(list(test_banks))).drop("CERT")
print(f"train {train.height:,} rows / test {test.height:,} rows "
      f"({len(test_banks)} held-out banks)")


train 5,174 rows / test 1,126 rows (687 held-out banks)


## Preliminary screening: native, neural and dual engines

Each engine is screening only: importance ranks, partial-dependence shape
hints, and pair nominations. Nothing here adopts a term.


In [4]:
probe = open_search(
    train, "deposit_rate", family=mb.Gaussian(), score="mse",
    folds=4, seed=7, mstop=120, nu=0.3, budget=CostBudget(fits=40),
)
probe_reports = {}
for engine in ("native", "neural", "dual"):
    ev = probe.run("pre_scan", engine=engine, rounds=120)
    report = ev.value
    probe_reports[engine] = report
    print(f"\n=== {engine}: training R²={report['training_r2']:.3f}, "
          f"fits={report['fits']} ===")
    print("priority:", report["priority"])
    print("top pairs:", report["pairs"][:4])
    if report.get("neural_failure"):
        print("neural_failure:", report["neural_failure"])

dual = probe_reports["dual"]
print("\nDual shape hints (partial dependence):")
for row in dual["columns"]:
    part = row["partial"]
    print(f"  {row['column']:18s} reason={row['reason']:10s} "
          f"shapes={row['shapes']}  "
          f"linR2={part['linear_r2']:.2f} curveR2={part['curve_r2']:.2f} "
          f"step={part['step_share']:.2f}")



=== native: training R²=0.904, fits=1 ===
priority: ['year', 'ib_share', 'loan_ratio', 'log_assets', 'brokered_share', 'offices', 'deposit_funding']
top pairs: [('year', 'log_assets', 0.014958044382090748, 0.008995064527604459), ('year', 'brokered_share', 0.012732856365036044, 0.007066006480731823), ('year', 'loan_ratio', 0.007324644077332668, 0.004284609927917016), ('ib_share', 'offices', 0.11835154302696746, 0.0030189176742598323)]



=== neural: training R²=0.813, fits=1 ===
priority: ['year', 'log_assets', 'offices', 'loan_ratio', 'ib_share', 'brokered_share', 'deposit_funding']
top pairs: [('year', 'log_assets', 0.023115732826748798, 0.013582939652280863), ('log_assets', 'offices', 1.0, 0.013047722640822258), ('year', 'offices', 0.012966644399568255, 0.007586151496807783), ('year', 'ib_share', 0.011778041860043853, 0.0066497671519203235)]



=== dual: training R²=0.904, fits=2 ===
priority: ['year', 'log_assets', 'offices', 'loan_ratio', 'ib_share', 'brokered_share', 'deposit_funding']
top pairs: [('year', 'log_assets', 0.023115732826748798, 0.013582939652280863), ('log_assets', 'offices', 1.0, 0.013047722640822258), ('year', 'offices', 0.012966644399568255, 0.007586151496807783), ('year', 'brokered_share', 0.012732856365036044, 0.007066006480731823)]

Dual shape hints (partial dependence):
  year               reason=ambiguous  shapes=['bols', 'bbs', 'btree']  linR2=0.68 curveR2=1.00 step=0.71
  log_assets         reason=ambiguous  shapes=['bols', 'bbs', 'btree']  linR2=0.84 curveR2=0.90 step=0.00
  loan_ratio         reason=ambiguous  shapes=['bols', 'bbs', 'btree']  linR2=0.88 curveR2=0.97 step=0.00
  brokered_share     reason=ambiguous  shapes=['bols', 'bbs', 'btree']  linR2=0.85 curveR2=0.96 step=0.00
  deposit_funding    reason=ambiguous  shapes=['bols', 'bbs', 'btree']  linR2=0.24 curveR2=0.46 step=0.52
  ib_share 

## Preliminary variable and interaction rankings

Across native, neural and dual, **`year` is first**. That is the Fed cycle
(ZIRP → hiking → pause) stamped onto every Call Report, not a bank choice.
Dual then ranks **`log_assets`**, **`offices`**, loan mix and funding mix.
Pair nominations concentrate on **`year × structure`** (size, brokered share,
loan ratio, offices) and on **`log_assets × offices`** (size vs branch
franchise). Shape hints mark size and several structure terms as
`ambiguous` — worth comparing `bols` / `bbs` / `btree`, not worth believing
from the probe alone.

Agreement is imperfect: neural sometimes saturates an \(H^2\) near 1 on a
pair the tree rates lower. Dual reports both scores; `compare` subsequently measures whether the proposed
interaction improves validation error.


## Banking interpretations and candidate hypotheses

**Symptoms vs causes.** A model that “discovers year” has rediscovered the
macro rate path. Treating year as a *cause of bank pricing* confuses the
environment with the franchise. The useful question is cross-sectional: given
the year, which banks pay more?

**Expected monotonic pieces.**

- `brokered_share` ↑ → deposit rate ↑ (wholesale / hot money).
- `offices` ↑ → deposit rate ↓ (sticky retail franchise), once size is held fixed.
- `loan_ratio` ↑ can raise funding need and willingness to pay up for deposits.

**Local bumps and missing variables.** Size (`log_assets`) mixes wholesale
access, pricing power, and business-model mix. A smooth with wiggles after
year is controlled often stands in for things this extract does not carry:
local deposit-market concentration, examiner ratings, uninsured-deposit
share, or TBTF funding channels. Those are hypotheses for later data, not
excuses to keep an unmeasured wiggle.

**Interaction hypothesis.** Fed-cycle passthrough should differ by franchise:
wholesale banks reprice faster than branch-heavy community banks. That
predicts `year × log_assets` / `year × offices` / `year × brokered_share`
structure — nominated by the probes, still unproven until `compare`.


In [5]:
session = open_search(
    train, "deposit_rate", family=mb.Gaussian(), score="mse",
    folds=4, seed=7, mstop=120, nu=0.3, budget=CostBudget(fits=250),
)
session.run("classify")
session.run("column_screen")
session.run(
    "hypothesis",
    statement=("After controlling for the calendar rate environment, banks with "
               "a higher brokered-deposit share pay a higher average rate on "
               "interest-bearing deposits."),
    variable="brokered_share", role="funding_mix",
    would_count_against=("A non-positive partial effect of brokered_share once "
                         "year and size are in the model."),
)
session.run(
    "hypothesis",
    statement=("More domestic offices mark a retail franchise that can fund "
               "without matching wholesale rates, so offices associate with a "
               "lower deposit rate after size is held fixed."),
    variable="offices", role="franchise",
    would_count_against="Offices raise rates once size is controlled.",
)

SearchEvidence(action='hypothesis', status='screening_only', evidence_id='0fd811ebf7a8ac37c59b154fb7fcf300c2ceba65678801cada1051857eefa90f', state_id='d1d17b2695bb2d34205b61edebd50c6a32404ac5c7b3c42c52495f049ef67280', fit_charges=0, seconds=1.9249971956014633e-05, summary={'statement': 'More domestic offices mark a retail franchise that can fund without matching wholesale rates, so offices associate with a lower deposit rate after size is held fixed.', 'variable': 'offices', 'role': 'franchise', 'would_count_against': 'Offices raise rates once size is controlled.', 'status': 'analyst-stated; not established by data'})

In [6]:
session.run(
    "hypothesis",
    statement=("Fed-cycle passthrough differs by bank size and franchise rather "
               "than shifting every bank equally."),
    variable="year", role="environment_interaction",
    would_count_against=("year×structure pairs fail a two-SE improvement over "
                         "additive year + structure terms."),
)

SearchEvidence(action='hypothesis', status='screening_only', evidence_id='05bf7624a85c4c7000d7df353a6935ca2c030c7ba068fe4803d508f2b8b554a1', state_id='1fcefb81e5978d8777128f91b55cf13d8f50a53cd38ccd94292b6769c4554447', fit_charges=0, seconds=4.2166910134255886e-05, summary={'statement': 'Fed-cycle passthrough differs by bank size and franchise rather than shifting every bank equally.', 'variable': 'year', 'role': 'environment_interaction', 'would_count_against': 'year×structure pairs fail a two-SE improvement over additive year + structure terms.', 'status': 'analyst-stated; not established by data'})

In [7]:
baseline = (
    "bbs(year, df=4) + bols(brokered_share) + bols(offices) "
    "+ bbs(log_assets, df=4) + bols(loan_ratio) + bols(ib_share) "
    "+ bols(deposit_funding)"
)
session.run("set_spec", rhs=baseline)
session.run("lock", columns="year")  # environment control stays
print("locked year; current spec:\n", session.spec)

locked year; current spec:
 bbs(year, df=4) + bols(brokered_share) + bols(offices) + bbs(log_assets, df=4) + bols(loan_ratio) + bols(ib_share) + bols(deposit_funding)


## Specification comparisons with `SearchSession`

Start additive with `year` locked. Compare a linear size alternative and a
smooth offices alternative. Adopt only under the two paired-SE rule. Then fit,
screen residuals, nominate pairs and a tree, and compare those nominations
the same way.


In [8]:
def clears(row, se_mult=2.0):
    return row["change"] < 0 and abs(row["change"]) > se_mult * row["change se"]

cmp_shapes = session.run(
    "compare",
    candidates={
        "linear_size": baseline.replace("bbs(log_assets, df=4)", "bols(log_assets)"),
        "smooth_offices": baseline.replace("bols(offices)", "bbs(offices, df=4)"),
    },
    mstop=120,
)
print("Shape compare (change = candidate − current MSE; negative helps):")
for row in cmp_shapes.summary["rows"]:
    flag = "ADOPT?" if row["specification"] != "current" and clears(row) else ""
    print(f"  {row['specification']:16s}  risk={row['out-of-fold risk']:.4f}  "
          f"Δ={row['change']:+.4f}  SE={row['change se']:.4f}  {flag}")

for row in cmp_shapes.summary["rows"]:
    if row["specification"] != "current" and clears(row):
        session.run("adopt", comparison_id=cmp_shapes.evidence_id,
                    candidate=row["specification"])
        print("adopted", row["specification"])
        break

# If smooth_offices still pending against a new baseline, measure it alone.
if "bbs(offices" not in session.spec:
    cand = session.spec.replace("bols(offices)", "bbs(offices, df=4)")
    cmp2 = session.run("compare", candidates={"smooth_offices": cand}, mstop=120)
    for row in cmp2.summary["rows"]:
        if row["specification"] != "current" and clears(row):
            session.run("adopt", comparison_id=cmp2.evidence_id,
                        candidate=row["specification"])
            print("adopted", row["specification"])


Shape compare (change = candidate − current MSE; negative helps):
  current           risk=0.2456  Δ=+0.0000  SE=0.0000  
  linear_size       risk=0.2457  Δ=+0.0001  SE=0.0001  
  smooth_offices    risk=0.2436  Δ=-0.0020  SE=0.0003  ADOPT?
adopted smooth_offices


In [9]:
session.run("fit")
ix = session.run("interaction_screen")
print("Residual interaction screen (top pairs):")
for row in ix.summary["top_pairs"]:
    print(f"  {row['left']} × {row['right']}: removed={row['removed']:.4f}")

# Reuse the dual report; surrogate=True would fit one fresh native surrogate.
pairs = session.run("pair_candidates", surrogate=dual)
print("\nPair nominations:")
for lead in pairs.summary["leads"]:
    print(f"  [{lead['source']:12s}] {lead['term']}")

tree = session.run("tree_candidate")
print("\nTree nomination:", tree.summary["term"])

Residual interaction screen (top pairs):
  year × loan_ratio: removed=0.0041
  brokered_share × ib_share: removed=0.0027
  loan_ratio × deposit_funding: removed=0.0025
  log_assets × brokered_share: removed=0.0024
  brokered_share × offices: removed=0.0024
  loan_ratio × brokered_share: removed=0.0021



Pair nominations:
  [residual    ] bbs(year, df=2.5, knots=5) %X% bbs(brokered_share, df=2.5, knots=5)
  [residual    ] bbs(year, df=2.5, knots=5) %X% bbs(loan_ratio, df=2.5, knots=5)
  [residual    ] bbs(year, df=2.5, knots=5) %X% bbs(log_assets, df=2.5, knots=5)
  [residual    ] bbs(brokered_share, df=2.5, knots=5) %X% bbs(offices, df=2.5, knots=5)
  [residual    ] bbs(loan_ratio, df=2.5, knots=5) %X% bbs(brokered_share, df=2.5, knots=5)
  [residual    ] bbs(year, df=2.5, knots=5) %X% bbs(ib_share, df=2.5, knots=5)
  [tree+neural ] bbs(log_assets, df=2.5, knots=5) %X% bbs(offices, df=2.5, knots=5)

Tree nomination: btree(year, log_assets, loan_ratio, brokered_share, deposit_funding, ib_share, offices, maxdepth=5, selection=sse)


In [10]:
interaction_candidates = {}
for lead in pairs.summary["leads"][:3]:
    term = lead["term"]
    if term and term not in session.spec:
        interaction_candidates[f"pair_{len(interaction_candidates)+1}"] = f"{session.spec} + {term}"
if tree.summary.get("term") and tree.summary["term"] not in session.spec:
    interaction_candidates["tree"] = f"{session.spec} + {tree.summary['term']}"
# Reasoned year×size tensor if not already present.
if "%X%" not in session.spec:
    interaction_candidates["year_by_size"] = (
        f"{session.spec} + bbs(year, df=2.5, knots=5) %X% bbs(log_assets, df=2.5, knots=5)"
    )

cmp_ix = session.run("compare", candidates=interaction_candidates, mstop=120)
print("\nInteraction compare:")
for row in cmp_ix.summary["rows"]:
    flag = "ADOPT?" if row["specification"] != "current" and clears(row) else ""
    print(f"  {row['specification']:14s}  risk={row['out-of-fold risk']:.4f}  "
          f"Δ={row['change']:+.4f}  SE={row['change se']:.4f}  {flag}")


Interaction compare:
  current         risk=0.2436  Δ=+0.0000  SE=0.0000  
  pair_1          risk=0.2388  Δ=-0.0048  SE=0.0015  ADOPT?
  pair_2          risk=0.2367  Δ=-0.0069  SE=0.0011  ADOPT?
  pair_3          risk=0.2381  Δ=-0.0055  SE=0.0006  ADOPT?
  tree            risk=0.2206  Δ=-0.0230  SE=0.0065  ADOPT?
  year_by_size    risk=0.2381  Δ=-0.0055  SE=0.0006  ADOPT?


In [11]:
ranked = cmp_ix.summary["two_se_candidates"]
print("Also cleared:", [row["candidate"] for row in cmp_ix.summary["also_cleared"]])
if ranked:
    # The caller chooses among measured alternatives; this replay chooses lowest MSE.
    winner = ranked[0]["candidate"]
    session.run("adopt", comparison_id=cmp_ix.evidence_id, candidate=winner)
    print("adopted", winner)


Also cleared: ['pair_2', 'pair_3', 'year_by_size', 'pair_1']
adopted tree


In [12]:
tr = session.run("term_report", shapes=True, mstop=120)
print("Term report (out-of-fold earn); tree terms can absorb additive signal:")
for row in tr.summary["terms"]:
    print(f"  {row['term'][:60]:60s}  earns={row['earns']:.4f}  SE={row['se']:.4f}")

tune = session.run("tune", start=120)
if session.model is None:
    session.run("fit")
print(f"\nTuned mstop={session.mstop}")
print("Final training specification:\n", session.spec)

pe = session.run("partial_effects",
                 which=["year", "brokered_share", "offices", "log_assets"])
print("partial_effects:", pe.summary)

session.run("freeze")
final = session.run("final_evaluate", data=test)
print(f"\nHeld-out banks: {len(test_banks)}; rows: {final.summary['n_test']}")
print(f"Final test MSE: {final.summary['score']:.4f} (percentage points)²")


/Users/fcela/src/public_github/mboostrs/python/mboostrs/_extrapolation.py:187: ExtrapolationWarning: 1 fold prediction in this call asked 1 learner(s) to predict outside the range they were fitted on. bbs(deposit_funding), covariate deposit_funding: 4 above 0.980861 (worst 0.063 training ranges out). A spline basis continues by its edge slope outside its boundary knots, which the fit does not constrain -- at 0.063 training ranges out that is an assumption, not an estimate. Pass term_report(extrapolation='allow') to accept it, or extrapolation='clip' to hold the effect flat outside the range instead, or extrapolation='error' to refuse. Guide: python -m mboostrs guide modeling-tactics.
  with routine_warnings(stacklevel=3, owner=fn.__name__):


Term report (out-of-fold earn); tree terms can absorb additive signal:
  btree(year, log_assets, loan_ratio, brokered_share, deposit_  earns=0.0230  SE=0.0065
  bbs(year, df=4)                                               earns=0.0000  SE=0.0000
  bols(brokered_share)                                          earns=0.0000  SE=0.0000
  bbs(offices, df=4)                                            earns=0.0000  SE=0.0000
  bbs(log_assets, df=4)                                         earns=0.0000  SE=0.0000
  bols(loan_ratio)                                              earns=0.0000  SE=0.0000
  bols(ib_share)                                                earns=0.0000  SE=0.0000
  bols(deposit_funding)                                         earns=0.0000  SE=0.0000



Tuned mstop=22
Final training specification:
 bbs(year, df=4) + bols(brokered_share) + bbs(offices, df=4) + bbs(log_assets, df=4) + bols(loan_ratio) + bols(ib_share) + bols(deposit_funding) + btree(year, log_assets, loan_ratio, brokered_share, deposit_funding, ib_share, offices, maxdepth=5, selection=sse)
partial_effects: {'which': ['year', 'brokered_share', 'offices', 'log_assets'], 'scale': 'component link scale', 'interpretation': 'fitted effect, not a causal contrast'}

Held-out banks: 687; rows: 1126
Final test MSE: 0.2512 (percentage points)²


## Reading the measured outcome

On the checked-in transcript the shape compare **adopts smooth offices**
(small but two-SE clear improvement) and keeps a smooth for size — linear size
does not help. Among interaction nominations, several `year × structure`
tensors clear the rule, and the multiway `btree(...)` clears by more; the
session adopts the tree because the compare ranks it first under the stated
heuristic.

The residual screen and dual probe suggested year×brokered and year×loan-ratio
surfaces. The multiway tree can fit these changing associations along with
other thresholds and interactions. An analyst
who wants an interpretable surface can still prefer the tensor that cleared
and reject the tree on substantive grounds — but that is a second, explicit
decision, not something the probe was allowed to force.

Holdout MSE on never-seen banks sits near the training out-of-fold risk of
the additive model, so the search did not obviously overfit the bank split.
`year` remains a locked environment control. The fitted terms describe
bank-level differences conditional on the Fed cycle; they do not estimate
a causal effect of year on deposit costs.

Reproduce the full transcript with:

The command below runs from `mboostrs-dev`; its experimental driver is not
part of the public package checkout.

```bash
RAYON_NUM_THREADS=1 .venv/bin/python tools/run_fdic_preexplore_reason.py
```


In [13]:
# Display the checked-in live-run transcript summary (provenance for the book).
if TRANSCRIPT.exists():
    art = json.loads(TRANSCRIPT.read_text())
    print("Transcript dataset:", art["dataset"]["train_rows"], "train /",
          art["dataset"]["test_rows"], "test")
    print("Probe priorities:")
    for eng, block in art["probes"].items():
        print(f"  {eng}: {block['report']['priority']}")
    print("Adopted:", art["search"]["adopted"])
    print("Final spec:", art["search"]["final_spec"])
    print("Final mstop:", art["search"]["final_mstop"])
    print("Holdout MSE:", art["search"]["final_evaluate"]["score"])
else:
    print("No checked-in transcript at", TRANSCRIPT)


Transcript dataset: 5174 train / 1126 test
Probe priorities:
  native: ['year', 'ib_share', 'loan_ratio', 'log_assets', 'brokered_share', 'offices', 'deposit_funding']
  neural: ['year', 'log_assets', 'offices', 'loan_ratio', 'ib_share', 'brokered_share', 'deposit_funding']
  dual: ['year', 'log_assets', 'offices', 'loan_ratio', 'ib_share', 'brokered_share', 'deposit_funding']
Adopted: ['smooth_offices', 'tree']
Final spec: bbs(year, df=4) + bols(brokered_share) + bbs(offices, df=4) + bbs(log_assets, df=4) + bols(loan_ratio) + bols(ib_share) + bols(deposit_funding) + btree(year, log_assets, loan_ratio, brokered_share, deposit_funding, ib_share, offices, maxdepth=5, selection=sse)
Final mstop: 22
Holdout MSE: 0.2511906834371538


## Relation to prior agent cases

[](cz_agent_search.ipynb) and [](agent_specification_search.ipynb) already
show one-action-at-a-time `SearchSession` discipline. [](preexploration_shapes.ipynb)
shows `autospec(..., preexploration=...)` shape pruning on a small classic
frame. This case applies those tools to bank-level records: inspect the
native/neural/dual rankings, state hypotheses about funding cost, and compare
the resulting candidate formulas on shared folds. Calendar-year effects adjust
for the rate environment while the bank characteristics describe differences
within it.
